In [1]:
!pip install peft transformers torch

  Using cached peft-0.21.2-py3-none-any.whl.metadata (15 kB)
  Using cached transformers-5.20.1-py3-none-any.whl.metadata (31 kB)
  Using cached torch-2.14.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (39 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached regex-2026.9.29-cp314-cp314-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached typer-0.27.3-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-4.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using 

In [2]:
import json
import torch
from huggingface_hub import hf_hub_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# The adapter is a CAUSAL_LM LoRA (no classification head): it was trained to
# answer with one word, so load it as a causal LM, not for sequence classification.
base_model_id = "Qwen/Qwen3.5-0.8B"
adapter_id = "regolo/brick-complexity-extractor"

# System prompt and user template used during training
meta = json.load(open(hf_hub_download(adapter_id, "training_metadata.json")))

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(model, adapter_id)
model.eval()

labels = ["easy", "medium", "hard"]
label_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in labels]

def classify(query):
    messages = [
        {"role": "system", "content": meta["system_prompt"]},
        {"role": "user", "content": meta["user_template"].format(query=query)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        next_token_logits = model(**inputs).logits[0, -1]
    # Compare the scores of the three label tokens only
    probs = next_token_logits[label_ids].softmax(-1)
    return labels[probs.argmax().item()], dict(zip(labels, probs.tolist()))

/Users/basil/GitHub/ibmhack26/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1017 15:43:19.267000 2566 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 9813.03it/s]


In [27]:
# Classify a query
query = "Build a football website."
predicted, probs = classify(query)
print(f"Complexity: {predicted}  {probs}")

Complexity: hard  {'easy': 1.6570091247558594e-05, 'medium': 0.004608154296875, 'hard': 0.99609375}


In [28]:
import joblib

# Load the TF-IDF + SGDClassifier pipeline trained on sh0416/wildchat-1m-tagged
# (see train_complexity_model.py). Predicts a complexity score from 1 (easy) to
# 10 (hard) based on the wording of a single user message.
wildchat_model = joblib.load("data/complexity_model.joblib")


In [29]:
def classify_wildchat_complexity(query):
    pred = wildchat_model.predict([query])[0]
    proba = wildchat_model.predict_proba([query])[0]
    classes = wildchat_model.named_steps["clf"].classes_
    probs = dict(sorted(zip(classes.tolist(), proba.tolist())))
    return int(pred), probs

query = "Build a football website."
pred, probs = classify_wildchat_complexity(query)
print(f"Complexity: {pred}")
print({k: round(v, 3) for k, v in probs.items()})


Complexity: 1
{1: 0.219, 2: 0.061, 3: 0.116, 4: 0.101, 5: 0.066, 6: 0.155, 7: 0.104, 8: 0.039, 9: 0.069, 10: 0.069}
